```
df = get-all-reports
for each cluster-id of neuron in question:
  output-dir = create-output-dir, add the cluster combined.jpeg file to this report
  for each image-key:
    for each position with cluster-id of our neuron:
      receptive field = find from (neuron, position)
      for each spatial position in the receptive field:
        spatial-out-dir = create output-dir / spatial-position
        dependent-neuron-and-cluster-ids = find the cluster-ids in df with our image-key and this spatial position, of neurons in the parent layer
        for each dep-neuron-and-cluster-id:
          spatial-out-dir / dep-neuron-and-cluster-id.jpeg <- copy the combined jpeg
```


Ohk, there is more to the story here. who activated on what point, is also important.  
So lets say that a spatial position `[0,1]` has some dependency `neuron0,chan0,cid0`.  
in this case, `[0,1]` is actually the input of multiple output points, we would like to know, which did it activate on?   

So we do need to get the clusters of the individual weights. in this case, we would have their own clusters, and their reports.  
For checking:

- some output point has some cluster. it also has a set of clusters on each relative position on its patch (if detected). If yes, then we try to see what the dependency was.
- So first, i need the reports of those clusters too (and they would need a new attribute for rel-position).
- we simply store the same y-position and x-position in these too (the output coord only, same as the original df for that neuron)
- for a given position, we query all rel-dfs to see if they have caught anythiung, if they did, get their rel-position, and then query the original df with those positions for dependencies. This gives us a new set of clusters, which are then concatnated to their rel pos. This seems  very principled.
- First though, ill need to add support for spatial positions in the code. its not there yet.  

In [ ]:
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import shutil
import matplotlib.pyplot as plt
from lucent.modelzoo import inceptionv1

plt.style.use("dark_background")

In [ ]:
#  might want to do this for new data.

! mkdir weight-banding
! aws s3 sync s3://narang99-private/lucent-workdir/mass-train-reports/mixed5b_5x5_bottleneck_pre_relu_conv ./weight-banding/mixed5b_5x5_bottleneck_pre_relu_conv
! aws s3 sync s3://narang99-private/lucent-workdir/mass-train-reports/mixed5b_5x5_pre_relu_conv/ ./weight-banding/mixed5b_5x5_pre_relu_conv
! aws s3 sync s3://narang99-private/lucent-workdir/mass-spatial-pos-train-reports/ weight-banding/_spatial_reports/


In [ ]:
import json
import os
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd
from PIL import Image
from tqdm import tqdm


def _receptive_block(i, ksize, stride, padding, input_size=None):
    """
    Returns [start, end) input indices (end=exclusive) that influence
    output position i of a conv layer.
    """
    start = i * stride - padding
    end = start + ksize

    if input_size is not None:
        start = max(start, 0)
        end = min(end, input_size)

    return start, end


def get_cluster_photo(
    base_report_dir,
    layer_name,
    channel,
    cluster_label,
    kind="combined",
    crop_max_height=None,
):
    report_dir = base_report_dir / layer_name / str(channel)
    jpeg = report_dir / f"cluster_{cluster_label}_{kind}.jpeg"
    if not jpeg.exists():
        if not report_dir.exists():
            raise Exception(f"report dir {report_dir} does not exist")
        print(f"file {jpeg} does not exist, it might be a singleton cluster")
        return
    if crop_max_height is not None:
        image = crop_top(jpeg, crop_max_height)
    else:
        image = Image.open(jpeg)
    return image


def crop_top(image_path, max_height):
    """
    Crops an image to at most `max_height` pixels from the top.
    If the image is already shorter than max_height, it's left unchanged.
    """
    img = Image.open(image_path)
    width, height = img.size

    if height <= max_height:
        # Image is already smaller (or equal) — no cropping needed
        cropped = img
    else:
        # box = (left, upper, right, lower)
        cropped = img.crop((0, 0, width, max_height))

    return cropped


def get_pos_by_def(layer_name, channel):
    # return the spatial position by df for the given layer and channel
    pos_by_df = {}
    for rel_x in tqdm(range(5)):
        for rel_y in range(5):
            p = Path(
                f"weight-banding/_spatial_reports/{rel_y}/{rel_x}/{layer_name}/{channel}/report.csv"
            )
            if p.exists():
                df = pd.read_csv(p)
                df = df[df.cluster_label != -1]
                pos_by_df[(rel_y, rel_x)] = df
    return pos_by_df


def _save_photo(src_dir, dest_dir, layer_name, chan, cid, spatial_cid=""):
    pil = get_cluster_photo(src_dir, layer_name, chan, cid, "combined", 490)
    if pil is None:
        return False
    pil.save(dest_dir / f"{layer_name}_{chan}_{spatial_cid}_{cid}_combined.jpeg")
    return True


def download_all_spatial_reports():
    ! aws s3 sync s3://narang99-private/lucent-workdir/mass-spatial-pos-train-reports/ weight-banding/_spatial_reports/

def untar_every_chan_tgz_if_not_done():
    
    for layer_dir in [d for d in report_base_dir.glob("*") if d.is_dir()]:
        chan_tgzs = [d for d in layer_dir.glob("*.tgz")]
        for chan_tgz in tqdm(chan_tgzs, desc=layer_dir.name):
            if not (chan_tgz.parent / chan_tgz.stem).exists():
                ! cd {chan_tgz.parent} && tar -xf {chan_tgz.name}

def untar_spatial_reports(layer_name, channel):
    for rel_x in tqdm(range(5)):
        for rel_y in range(5):
            p = Path(f"weight-banding/_spatial_reports/{rel_y}/{rel_x}/{layer_name}/{channel}.tgz")
            if p.exists() and not (p.parent / p.stem).exists():
                ! cd {p.parent} && tar -xf {p.name}

def get_all_dfs(report_base_dir):
    # get all csv files of a given layer
    # we work with whatever we have, at least some prelimnary evidence hopefully
    csv_files = list(report_base_dir.rglob("report.csv"))
    dfs = []
    for f in tqdm(csv_files):
        df = pd.read_csv(f)
        df = df[df.cluster_label != -1].reset_index()
        dfs.append(df)
    pdf = pd.concat(dfs)
    return pdf




def get_pos_by_found_results(
    model,
    layer_name,
    channel,
    curr_cluster_id,
    pdf,
    pos_by_df,
):
    # for each cluster, we pick 5 positions, and create reports
    layer = model.get_submodule(layer_name)
    y_max, x_max = layer.kernel_size

    # first save one photo

    curr_cluster_df = pdf[
        (pdf.cluster_label == curr_cluster_id)
        & (pdf.layer_name == layer_name)
        & (pdf.channel == channel)
    ]
    ikeys = curr_cluster_df.input_image_key.unique()

    pos_by_results = defaultdict(list)

    for ikey in tqdm(ikeys):
        curr_cluster_positions_df = curr_cluster_df[
            curr_cluster_df.input_image_key == ikey
        ][["y_position", "x_position"]]
        positions = [
            (t.y_position, t.x_position) for t in curr_cluster_positions_df.itertuples()
        ]
        for y, x in positions:
            y0, y1 = _receptive_block(y, layer.kernel_size[0], layer.stride[0], 2)
            x0, x1 = _receptive_block(x, layer.kernel_size[1], layer.stride[1], 2)
            for rel_y in range(y_max):
                for rel_x in range(x_max):
                    cur_y, cur_x = y0 + rel_y, x0 + rel_x
                    spatial_df = pos_by_df.get((rel_y, rel_x))
                    if spatial_df is None:
                        continue
                    filtered_spatial_df = spatial_df[
                        (spatial_df.y_position == cur_y)
                        & (spatial_df.x_position == cur_x)
                        & (spatial_df.cluster_label != -1)
                        & (spatial_df.input_image_key == ikey)
                    ]

                    if len(filtered_spatial_df) > 0:
                        # we detected something here
                        assert len(filtered_spatial_df) == 1
                        spatial_cluster_label = next(
                            filtered_spatial_df.itertuples()
                        ).cluster_label

                        # only find in dependency layer
                        deps_on_pos_df = pdf[
                            (pdf.y_position == cur_y)
                            & (pdf.x_position == cur_x)
                            & (pdf.layer_name == "mixed5b_5x5_bottleneck_pre_relu_conv")
                            & (pdf.input_image_key == ikey)
                        ]
                        for tup in deps_on_pos_df.itertuples():
                            pos_by_results[(rel_y, rel_x)].append(
                                (
                                    tup.layer_name,
                                    tup.channel,
                                    tup.cluster_label,
                                    spatial_cluster_label,
                                )
                            )

    return pos_by_results


def _sanitised(s):
    return "".join([(c if c.isalnum() else "_") for c in s])


def save_stage_2_report(curr_res, pos_by_res, report_base_dir, dest_dir):
    pos_by_res = {p: set(r) for p, r in pos_by_res.items()}
    layer_name, channel, curr_cluster_id = curr_res

    _save_photo(report_base_dir, dest_dir, layer_name, channel, curr_cluster_id)

    for pos, res in tqdm(pos_by_res.items()):
        dest = dest_dir / _sanitised(str(pos))
        dest.mkdir(parents=True, exist_ok=True)
        for layer_name, channel, cluster_id, spatial_cluster_id in res:
            _save_photo(
                report_base_dir,
                dest,
                layer_name,
                channel,
                cluster_id,
                spatial_cluster_id,
            )


def generate_cluster_report_markdown(cluster_dir, base_dir) -> str:
    """
    Generate Quarto markdown for a single cluster's report directory.

    Parameters
    ----------
    cluster_dir : str
        Path to the directory for a single cluster (contains the main neuron's
        top-level jpeg, spatial position subfolders, and meta.json).
    base_dir : str
        Name of the base directory the generated image paths should be
        relative to (prepended to every image src).
    """
    cluster_dir = Path(cluster_dir)

    # --- main neuron photo (top-level jpeg only) ---
    main_photo = next(
        (f for f in cluster_dir.glob("*_combined.jpeg") if f.parent == cluster_dir),
        None,
    )
    if main_photo is None:
        raise FileNotFoundError(f"No main neuron photo found in {cluster_dir}")

    m = re.match(r"(.+)_(\d+)__(\d+)_combined\.jpeg$", main_photo.name)
    if not m:
        raise ValueError(f"Could not parse main photo filename: {main_photo.name}")
    layer_name, channel, cluster_id = m.groups()

    lines = ["::: {.scrollable-tabset}\n", "::: {.panel-tabset}\n", "## Main neuron\n"]
    lines.append(
        f"![`{layer_name}` channel {channel}, cluster {cluster_id}]"
        f"({base_dir}/{cluster_dir.name}/{main_photo.name})\n"
    )

    # --- spatial position folders ---
    spatial_dirs = sorted(
        (
            d
            for d in cluster_dir.iterdir()
            if d.is_dir() and d.name != ".ipynb_checkpoints"
        ),
        key=lambda d: tuple(int(n) for n in re.findall(r"\d+", d.name)),
    )

    for d in spatial_dirs:
        pos = tuple(int(n) for n in re.findall(r"\d+", d.name))
        lines.append(f"## Position {pos}\n")
        for f in sorted(d.glob("*_combined.jpeg")):
            pm = re.match(r"(.+)_(\d+)_(\d+)_(\d+)_combined\.jpeg$", f.name)
            if not pm:
                continue
            p_layer, p_chan, _spatial_cid, p_cid = pm.groups()
            caption = f"{p_layer}:{p_chan}, cluster {p_cid}"
            lines.append(
                f"![{caption}]({base_dir}/{cluster_dir.name}/{d.name}/{f.name})\n"
            )
        lines.append("")

    lines.append(":::\n")  # tabset

    lines.append(":::\n")  # scrollable

    return "\n".join(lines)


def mark_done(directory_path, done=True):
    os.makedirs(directory_path, exist_ok=True)
    meta_path = os.path.join(directory_path, "meta.json")

    if os.path.isfile(meta_path):
        with open(meta_path, "r") as f:
            meta = json.load(f)
    else:
        meta = {}

    meta["done"] = bool(done)

    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    return meta["done"]


def is_done(directory_path):
    meta_path = os.path.join(directory_path, "meta.json")

    if not os.path.isdir(directory_path) or not os.path.isfile(meta_path):
        return False

    with open(meta_path, "r") as f:
        meta = json.load(f)

    return bool(meta.get("done", False))


def prepare_reports_for_all_cids(
    model,
    layer_name,
    channel,
    neuron_df,
    deps_df,
    pos_by_df,
    cids,
    root_dest_dir,
    report_base_dir,
):
    dest_dir = root_dest_dir / layer_name / str(channel)
    for n, cid in enumerate(cids):
        print(
            f"############################### {n}/{len(cids)} #################################"
        )
        ddir = dest_dir / str(cid)
        if is_done(ddir):
            print(f"SKIP: already done. {cid}")
            continue
        pdf = pd.concat([deps_df, neuron_df])
        pos_by_res = get_pos_by_found_results(
            model, layer_name, channel, cid, pdf, pos_by_df
        )

        ddir.mkdir(parents=True, exist_ok=True)
        save_stage_2_report(
            (layer_name, channel, cid), pos_by_res, report_base_dir, ddir
        )

        mark_done(ddir)

In [ ]:
untar_every_chan_tgz_if_not_done()

In [ ]:
report_base_dir = Path("weight-banding")
ROOT_DEST_DIR = report_base_dir / "STAGE3_REPORTS"
ROOT_DEST_DIR.mkdir(parents=True, exist_ok=True)

device = "cpu"
model = inceptionv1(pretrained=True)
model = model.to(device)
model = model.eval()


In [ ]:
deps_df = get_all_dfs(report_base_dir / "mixed5b_5x5_bottleneck_pre_relu_conv")

In [ ]:
cids = [22, 18, 21]
layer_name = "mixed5b_5x5_pre_relu_conv"
channel = 8

untar_spatial_reports(layer_name, channel)
pos_by_df = get_pos_by_def(layer_name, channel)
neuron_df = pd.read_csv(report_base_dir / layer_name / str(channel) / "report.csv")
print("REPORT GENERATION: start")
prepare_reports_for_all_cids(model, layer_name, channel, neuron_df, deps_df, pos_by_df, cids, ROOT_DEST_DIR, report_base_dir)

In [ ]:
cids = [17, 23, 22, 15, 10, 7, 12, 24, 16, 9, 14]
layer_name = "mixed5b_5x5_pre_relu_conv"
channel = 8

untar_spatial_reports(layer_name, channel)
pos_by_df = get_pos_by_def(layer_name, channel)
neuron_df = pd.read_csv(report_base_dir / layer_name / str(channel) / "report.csv")
print("REPORT GENERATION: start")
prepare_reports_for_all_cids(model, layer_name, channel, neuron_df, deps_df, pos_by_df, cids, ROOT_DEST_DIR, report_base_dir)

In [ ]:
cids = [17, 23, 22, 15, 10, 7, 12, 24, 16, 9, 14]
layer_name = "mixed5b_5x5_pre_relu_conv"
channel = 12

untar_spatial_reports(layer_name, channel)
pos_by_df = get_pos_by_def(layer_name, channel)
neuron_df = pd.read_csv(report_base_dir / layer_name / str(channel) / "report.csv")
print("REPORT GENERATION: start")
prepare_reports_for_all_cids(model, layer_name, channel, neuron_df, deps_df, pos_by_df, cids, ROOT_DEST_DIR, report_base_dir)

In [ ]:
cids = [21, 9, 2, 17, 19, 16, 8, 20, 0, 15, ]
layer_name = "mixed5b_5x5_pre_relu_conv"
channel = 13

untar_spatial_reports(layer_name, channel)
pos_by_df = get_pos_by_def(layer_name, channel)
neuron_df = pd.read_csv(report_base_dir / layer_name / str(channel) / "report.csv")
print("REPORT GENERATION: start")
prepare_reports_for_all_cids(model, layer_name, channel, neuron_df, deps_df, pos_by_df, cids, ROOT_DEST_DIR, report_base_dir)

# Visualise the weight

In [ ]:
layer_name = "mixed5b_5x5_pre_relu_conv"
channel = 12

In [ ]:
from olt.show import show_single_channel_red_green_black as S
from sklearn.cluster import KMeans
w = model.get_submodule(layer_name).weight[channel].detach().cpu()
chan_last_w = w.reshape(w.shape[0], -1).permute(1, 0).numpy() # [pos, chan]

_, axes = plt.subplots(1, 3, figsize=(15,5))

ins = [KMeans(c).fit(chan_last_w).inertia_ for c in range(1, 25)]
axes[0].plot(ins)

# ideal comps is not 5 though
km = KMeans(5).fit(chan_last_w)
axes[1].imshow(km.labels_.reshape(5,5), cmap="tab10")


km = KMeans(8).fit(chan_last_w)
axes[2].imshow(km.labels_.reshape(5,5), cmap="tab10")


plt.show()

S([w_.reshape(6,8) for w_ in chan_last_w], (3*5,3*5), ncols=5, suptitle="all spatial positions")
plt.show()